# Telco Customer Churn Prediction

**Author:** Blessing Osuagwu  
**Task:** Binary classification of customer churn using a public sample dataset.

The original notebook was a useful first exploration, but it evaluated on data that had also been used to fit preprocessing and models. This version creates the holdout split first and keeps preprocessing inside model pipelines to provide a more credible estimate of generalization.

## Dataset and target

The Kaggle Telco Customer Churn dataset contains customer account, service, and demographic fields. The target is `Churn`: `Yes` is the positive class (customer churns). This dataset represents a sample snapshot; it does not include the timing of a prospective prediction or intervention outcomes.


## 1. Setup and load data

Install dependencies from `requirements.txt`. Download the Kaggle CSV and place it in `data/` or `/content/` in Google Colab.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

SEED = 42
DATA_FILE = Path("data/Telecom_Churn.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path("/content/Telecom_Churn.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError("Download the Telco churn CSV linked in the README and place it in data/ or /content/.")

df = pd.read_csv(DATA_FILE)
print(f"Shape: {df.shape}")
display(df.head())

## 2. Data quality and target balance

The source CSV encodes `TotalCharges` as text and contains blank strings for some new customers. Convert that column to numeric; its missing values will be imputed inside each training fold by the model pipeline. Do not impute on the complete dataset before splitting.

In [ ]:
required = {"customerID", "Churn", "TotalCharges"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")

df = df.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["Churn"] = df["Churn"].astype(str).str.strip()
if not set(df["Churn"].dropna().unique()).issubset({"Yes", "No"}):
    raise ValueError("Unexpected values in Churn target")

print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).head(10))
print("Churn counts:")
display(df["Churn"].value_counts())
print("Churn rate:", round(df["Churn"].eq("Yes").mean(), 3))

In [ ]:
ax = df["Churn"].value_counts().reindex(["No", "Yes"]).plot(
    kind="bar", color=["#4c78a8", "#e45756"], title="Customer churn labels"
)
ax.set_xlabel("Churn"); ax.set_ylabel("Customers")
plt.tight_layout(); plt.show()

sns.boxplot(data=df, x="Churn", y="tenure", hue="Churn", legend=False)
plt.title("Tenure distribution by churn label"); plt.tight_layout(); plt.show()

## 3. Split before preprocessing

The test set is stratified to preserve the churn proportion. The customer identifier is excluded. Imputation, one-hot encoding, and scaling are fitted by pipelines using training data only. No SMOTE is applied; class weighting provides a simpler imbalance-aware starting point.

In [ ]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train churn rate:", round(y_train.mean(), 3), "Test churn rate:", round(y_test.mean(), 3))

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features),
])

## 4. Train baseline models

Both models use the same training/test split and training-only preprocessing. The models are baselines; compare recall and average precision for the minority churn class rather than choosing by accuracy alone.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample",
            random_state=SEED, n_jobs=-1,
        )),
    ]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: fitted")

## 5. Held-out evaluation

At the default 0.5 probability threshold, report recall, precision, F1, ROC-AUC, and average precision for the churn class. The threshold should ultimately reflect the relative costs of contacting a customer who would stay and missing a customer who would churn.

In [ ]:
rows = []
for name, model in models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "churn_precision": precision_score(y_test, pred, zero_division=0),
        "churn_recall": recall_score(y_test, pred, zero_division=0),
        "churn_f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, prob),
        "average_precision": average_precision_score(y_test, prob),
    })
    print("\n", name)
    print(classification_report(y_test, pred, target_names=["No churn", "Churn"], digits=3, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["No churn", "Churn"], cmap="Blues")
    plt.title(f"{name}: held-out test set")
    plt.tight_layout(); plt.show()

results = pd.DataFrame(rows).sort_values("average_precision", ascending=False)
display(results)

## 6. Interpretation and limitations

Use the test metrics to compare these baselines, then tune parameters or a decision threshold using cross-validation within the training set only. Do not repeatedly use the test set for model selection. A high score does not show that a retention campaign will work; that requires intervention and outcome data. The dataset is a public benchmark and may not reflect a current provider's customer population.